# Fraud Detection - Complete Modelling Notebook (PaySim)

Everything after the baseline, in one notebook. Run the cells top to bottom.

**What this notebook does**
1. Loads PaySim and keeps only `TRANSFER` and `CASH_OUT` rows (the only types that ever contain fraud)
2. Builds features. `FEATURE_MODE = 'pre'` (default) uses only information known before a transaction is approved. `FEATURE_MODE = 'full'` uses **all PaySim columns**, including the balances after the transaction
3. Splits the data 70 / 15 / 15 (train / validation / test), stratified
4. Trains 6 models: Logistic Regression, Random Forest, XGBoost (class weights), XGBoost + SMOTE, XGBoost + undersampling, Isolation Forest
   then **tunes XGBoost's hyperparameters with stratified cross-validation** and reports a 5-fold cross-validated score
5. Picks each model's threshold on the **validation** set, reports on the **test** set
6. Tunes a cost-based threshold (missed fraud vs false alarm)
7. Explains the best model with SHAP
8. Shows a **feature comparison**: basic features vs pre-transaction features vs full PaySim columns
9. Saves everything the Streamlit app needs into `fraud_artifacts.zip`

**Before running:** get the CSV into Colab (see the next cell for 3 ways).

In [ ]:
!pip -q install imbalanced-learn xgboost shap

import os, json, warnings, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, sklearn, xgboost

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (average_precision_score, precision_recall_curve, roc_curve,
                             precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay)
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTENC
from imblearn.under_sampling import RandomUnderSampler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
RS = 42
os.makedirs('figures', exist_ok=True)

## 1. Get the data into Colab

Pick **one** of these options. Option A is the most reliable if Google Drive mounting gives you trouble.

- **Option A (Kaggle API):** Kaggle -> Settings -> API -> *Create New Token* downloads `kaggle.json`. Upload it when prompted.
- **Option B (upload):** run `files.upload()` and pick the CSV (slow for a 470 MB file).
- **Option C (Drive):** mount Drive and point `CSV_PATH` at the file.

In [ ]:
CSV_PATH = '/content/PS_20174392719_1491204439457_log.csv'

# ---- Option A: Kaggle API (uncomment the 4 lines) ----
# from google.colab import files
# files.upload()   # upload kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d ealaxi/paysim1 -p /content --unzip

# ---- Option B: direct upload (uncomment) ----
# from google.colab import files
# files.upload()

# ---- Option C: Google Drive (uncomment, then set CSV_PATH to your Drive path) ----
# from google.colab import drive
# drive.mount('/content/drive')
# CSV_PATH = '/content/drive/MyDrive/PS_20174392719_1491204439457_log.csv'

assert os.path.exists(CSV_PATH), f'CSV not found at {CSV_PATH}. Use one of the options above first.'
print('Found:', CSV_PATH)

## 2. Load, keep risky types, build features

Fraud only ever occurs in `TRANSFER` and `CASH_OUT`, so the model is trained on those rows only. In the app, any other type is labelled legitimate automatically. This also raises the fraud rate in the training data from about 0.13% to about 0.3%.

**Two feature modes (set `FEATURE_MODE` in the next cell):**

- `'full'`: all PaySim columns, including `newbalanceOrig` and `newbalanceDest` (balances **after** the transaction). PaySim cancels fraudulent transactions, so these columns reveal the outcome and scores come out very high. This is how most published notebooks for this dataset report results.
- `'pre'` (default): only information known **before** a transaction is approved (amount, hour, type, balances before the transaction, and ratios or flags built from them). Scores are lower but reflect what a live system could do.

Section 9 trains both so you can show the comparison.

In [ ]:
USE = ['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
       'oldbalanceDest', 'newbalanceDest', 'isFraud']
raw = pd.read_csv(CSV_PATH, usecols=USE)
print('Full dataset:', raw.shape, '| fraud rate: {:.4%}'.format(raw['isFraud'].mean()))
print('\nFrauds by type:')
print(raw.groupby('type')['isFraud'].agg(['count', 'sum']).rename(columns={'count': 'rows', 'sum': 'frauds'}))

RISKY = ['TRANSFER', 'CASH_OUT']
df = raw[raw['type'].isin(RISKY)].reset_index(drop=True)
del raw
print('\nModelling rows:', df.shape, '| fraud rate: {:.4%}'.format(df['isFraud'].mean()))

In [ ]:
FEATURE_MODE = 'pre'   # 'pre' = only information known before approval | 'full' = all PaySim columns

df['hour'] = df['step'] % 24                      # step = 1 hour of simulation time
df['is_transfer'] = (df['type'] == 'TRANSFER').astype(int)

# Features known BEFORE a transaction is approved
df['amt_to_orig_bal'] = df['amount'] / (df['oldbalanceOrg'] + 1)     # share of the sender's balance being moved
df['amt_to_dest_bal'] = df['amount'] / (df['oldbalanceDest'] + 1)
df['drains_account'] = ((df['oldbalanceOrg'] > 0) & (df['amount'] >= df['oldbalanceOrg'] - 0.01)).astype(int)
df['orig_empty'] = (df['oldbalanceOrg'] == 0).astype(int)
df['dest_empty'] = (df['oldbalanceDest'] == 0).astype(int)         # receiver account empty before the transfer

# Features that exist only AFTER the transaction is processed (they reveal the outcome in PaySim)
df['errOrig'] = df['newbalanceOrig'] + df['amount'] - df['oldbalanceOrg']
df['errDest'] = df['oldbalanceDest'] + df['amount'] - df['newbalanceDest']

BASIC_FEATURES = ['amount', 'hour', 'is_transfer']
PRE_FEATURES = BASIC_FEATURES + ['oldbalanceOrg', 'oldbalanceDest', 'amt_to_orig_bal',
                                 'amt_to_dest_bal', 'drains_account', 'orig_empty', 'dest_empty']
POST_FEATURES = ['newbalanceOrig', 'newbalanceDest', 'errOrig', 'errDest']
FULL_FEATURES = PRE_FEATURES + POST_FEATURES

FEATURES = FULL_FEATURES if FEATURE_MODE == 'full' else PRE_FEATURES
BINARY_FEATURES = ['is_transfer', 'drains_account', 'orig_empty', 'dest_empty']
CAT_IDX = [FEATURES.index(c) for c in BINARY_FEATURES]   # used by SMOTENC

y = df['isFraud']
print('FEATURE_MODE =', FEATURE_MODE)
print('Features:', FEATURES)
if FEATURE_MODE == 'full':
    print('NOTE: this mode uses balances AFTER the transaction. State this in your report (see section 9).')

## 3. Train / validation / test split (stratified)

- **Train (70%)**: fit models (resampling happens only here)
- **Validation (15%)**: choose each model's decision threshold
- **Test (15%)**: final numbers for the report, touched once

The test set keeps the natural fraud rate, so the metrics are realistic.

In [ ]:
idx = df.index.values
idx_train, idx_tmp = train_test_split(idx, test_size=0.30, stratify=y, random_state=RS)
idx_val, idx_test = train_test_split(idx_tmp, test_size=0.50, stratify=y.loc[idx_tmp], random_state=RS)

X_train, X_val, X_test = df.loc[idx_train, FEATURES], df.loc[idx_val, FEATURES], df.loc[idx_test, FEATURES]
y_train, y_val, y_test = y.loc[idx_train], y.loc[idx_val], y.loc[idx_test]

for n, a in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    print(f'{n:5s}: {len(a):>9,} rows | frauds: {int(a.sum()):>5,} ({a.mean():.4%})')

## 4. Helper functions

`run_model` fits a model, picks the threshold that maximises F1 on the **validation** set, and then reports precision, recall, F1 and PR-AUC on the **test** set. PR-AUC does not depend on the threshold, so it is the fairest single number for comparing models.

In [ ]:
def best_f1_threshold(y_true, probs):
    p, r, t = precision_recall_curve(y_true, probs)
    f1 = 2 * p[:-1] * r[:-1] / np.maximum(p[:-1] + r[:-1], 1e-12)
    return float(t[np.argmax(f1)])

trained = {}     # name -> dict(model, pv, pt, thr)
rows = []

def run_model(name, model, Xtr, ytr, Xva, Xte, group='final'):
    model.fit(Xtr, ytr)
    pv = model.predict_proba(Xva)[:, 1]
    pt = model.predict_proba(Xte)[:, 1]
    thr = best_f1_threshold(y_val, pv)
    pred = (pt >= thr).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_test, pred, average='binary', zero_division=0)
    pr_auc = average_precision_score(y_test, pt)
    rows.append({'Model': name, 'Precision': p, 'Recall': r, 'F1': f1, 'PR-AUC': pr_auc,
                 'Threshold': thr, 'Feature set': group})
    trained[name] = {'model': model, 'pv': pv, 'pt': pt, 'thr': thr}
    print(f'{name:28s} PR-AUC={pr_auc:.4f}  P={p:.4f}  R={r:.4f}  F1={f1:.4f}  thr={thr:.4f}')

class IsoWrapper:
    '''Isolation Forest (unsupervised) exposed with a predict_proba interface.'''
    def __init__(self):
        self.m = IsolationForest(n_estimators=200, contamination=float(y_train.mean()),
                                 random_state=RS, n_jobs=-1)
    def fit(self, X, y=None):
        self.m.fit(X)
        s = -self.m.score_samples(X)
        self.lo, self.hi = s.min(), s.max()
        return self
    def predict_proba(self, X):
        s = (-self.m.score_samples(X) - self.lo) / (self.hi - self.lo + 1e-12)
        s = np.clip(s, 0, 1)
        return np.c_[1 - s, s]

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
spw = float(np.sqrt(neg / pos))   # softened class weight for XGBoost
print(f'scale_pos_weight used: {spw:.1f} (sqrt of the full imbalance ratio {neg/pos:.0f})')

def make_xgb(**kw):
    return XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8,
                         tree_method='hist', eval_metric='aucpr', n_jobs=-1,
                         random_state=RS, **kw)

## 5. Train the models

In [ ]:
# 1. Baseline: Logistic Regression
LOG_COLS = [c for c in ['amount', 'oldbalanceOrg', 'oldbalanceDest', 'amt_to_orig_bal', 'amt_to_dest_bal',
                        'newbalanceOrig', 'newbalanceDest'] if c in FEATURES]
SLOG_COLS = [c for c in ['errOrig', 'errDest'] if c in FEATURES]    # can be negative, so use a signed log

def signed_log1p(x):
    return np.sign(x) * np.log1p(np.abs(x))

parts = [('log', Pipeline([('f', FunctionTransformer(np.log1p)), ('sc', StandardScaler())]), LOG_COLS),
         ('hr', StandardScaler(), ['hour'])]
if SLOG_COLS:
    parts.append(('slog', Pipeline([('f', FunctionTransformer(signed_log1p)), ('sc', StandardScaler())]), SLOG_COLS))
lr_pre = ColumnTransformer(parts, remainder='passthrough')
run_model('Logistic Regression',
          Pipeline([('pre', lr_pre), ('m', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RS))]),
          X_train, y_train, X_val, X_test)

In [ ]:
# 2. Random Forest with class weights
run_model('Random Forest',
          RandomForestClassifier(n_estimators=150, max_depth=14, min_samples_leaf=5,
                                 class_weight='balanced_subsample', n_jobs=-1, random_state=RS),
          X_train, y_train, X_val, X_test)

In [ ]:
# 3. XGBoost with class weights
run_model('XGBoost (weighted)', make_xgb(scale_pos_weight=spw), X_train, y_train, X_val, X_test)

In [ ]:
# 4. XGBoost + SMOTE (SMOTENC because several features are binary flags). Training data only.
run_model('XGBoost + SMOTE',
          ImbPipeline([('smote', SMOTENC(categorical_features=CAT_IDX, sampling_strategy=0.1, random_state=RS)),
                       ('xgb', make_xgb())]),
          X_train, y_train, X_val, X_test)

In [ ]:
# 5. XGBoost + random undersampling (training data only)
run_model('XGBoost + undersampling',
          ImbPipeline([('under', RandomUnderSampler(sampling_strategy=0.1, random_state=RS)),
                       ('xgb', make_xgb())]),
          X_train, y_train, X_val, X_test)

In [ ]:
# 6. Isolation Forest (unsupervised anomaly detection, never sees labels)
run_model('Isolation Forest (unsupervised)', IsoWrapper(), X_train, y_train, X_val, X_test)

## 5b. Hyperparameter tuning with cross-validation

Default settings are rarely best. **Hyperparameter tuning** searches for the model settings (tree depth, learning rate, number of trees, regularisation, class weight) that score best on **stratified k-fold cross-validation**: the training data is split into folds that keep the same fraud rate, each fold is held out in turn, and the average PR-AUC across folds picks the winner.

- The search uses only the **training set** (a stratified sample of it, to keep it fast). Validation and test data are not touched, so there is no leakage.
- Raise `N_ITER` and `TUNE_FRAC` for a more thorough search if you have time. Lower them if Colab is slow.

In [ ]:
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_validate
from scipy.stats import randint, uniform, loguniform

N_ITER = 20          # random hyperparameter combinations to try
TUNE_FRAC = 0.15     # share of the training set used for the search
CV_FOLDS = 3         # folds inside the search
CV_FINAL_FOLDS = 5   # folds for the final cross-validated score
CV_FINAL_FRAC = 0.5  # share of the training set used for that final score (1.0 = all, slower)

X_tune, _, y_tune, _ = train_test_split(X_train, y_train, train_size=TUNE_FRAC, stratify=y_train, random_state=RS)
print(f'Tuning on {len(X_tune):,} rows ({int(y_tune.sum())} frauds), {CV_FOLDS}-fold CV, {N_ITER} combinations')

param_dist = {
    'n_estimators': randint(200, 601),
    'max_depth': randint(3, 10),
    'learning_rate': loguniform(0.02, 0.3),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'min_child_weight': randint(1, 20),
    'gamma': uniform(0, 5),
    'reg_lambda': loguniform(0.1, 20),
    'scale_pos_weight': [1.0, spw, float(neg / pos)],
}

def xgb_base(**kw):
    return XGBClassifier(tree_method='hist', eval_metric='aucpr', n_jobs=-1, random_state=RS, **kw)

search = RandomizedSearchCV(
    xgb_base(), param_distributions=param_dist, n_iter=N_ITER, scoring='average_precision',
    cv=StratifiedKFold(CV_FOLDS, shuffle=True, random_state=RS),
    n_jobs=1, random_state=RS, refit=False, verbose=1)
search.fit(X_tune, y_tune)

best_params = {k: (v.item() if hasattr(v, 'item') else v) for k, v in search.best_params_.items()}
tuning = (pd.DataFrame(search.cv_results_)
          [['rank_test_score', 'mean_test_score', 'std_test_score'] + ['param_' + k for k in param_dist]]
          .sort_values('rank_test_score').rename(columns={'mean_test_score': 'CV PR-AUC (mean)', 'std_test_score': 'CV PR-AUC (std)'}))
print('Best parameters:', best_params)
display(tuning.head(5).round(4))
tuning.round(4).to_csv('tuning_results.csv', index=False)

In [ ]:
# Retrain with the best parameters on the full training set, then evaluate on validation/test as before
run_model('XGBoost (tuned)', xgb_base(**best_params), X_train, y_train, X_val, X_test)

In [ ]:
# Cross-validated score of the tuned model (shows the result is stable, not a lucky split)
X_cv, _, y_cv, _ = train_test_split(X_train, y_train, train_size=CV_FINAL_FRAC, stratify=y_train, random_state=RS)     if CV_FINAL_FRAC < 1 else (X_train, None, y_train, None)
cv = cross_validate(xgb_base(**best_params), X_cv, y_cv,
                    cv=StratifiedKFold(CV_FINAL_FOLDS, shuffle=True, random_state=RS),
                    scoring={'PR-AUC': 'average_precision', 'ROC-AUC': 'roc_auc'}, n_jobs=1)
cv_folds = pd.DataFrame({'PR-AUC': cv['test_PR-AUC'], 'ROC-AUC': cv['test_ROC-AUC']},
                        index=[f'Fold {i+1}' for i in range(CV_FINAL_FOLDS)])
cv_summary = pd.concat([cv_folds, cv_folds.agg(['mean', 'std']).rename(index={'mean': 'Mean', 'std': 'Std'})]).round(4)
display(cv_summary)
cv_summary.to_csv('cv_summary.csv')

## 6. Results table

Fraud is rare, so PR-AUC and recall at a sensible precision matter, not accuracy. Random guessing would give a PR-AUC equal to the fraud rate.

In [ ]:
res = pd.DataFrame(rows)
main = res[res['Feature set'] == 'final'].drop(columns='Feature set').set_index('Model').sort_values('PR-AUC', ascending=False)
print('Random-guess PR-AUC (fraud rate in test set): {:.4f}'.format(y_test.mean()))
display(main.round(4))
main.round(4).to_csv('results_table.csv')

In [ ]:
# Precision-recall curves for all models + confusion matrix of the best one
fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))
for name in main.index:
    p, r, _ = precision_recall_curve(y_test, trained[name]['pt'])
    ax[0].plot(r, p, label=f"{name} ({main.loc[name, 'PR-AUC']:.3f})")
ax[0].axhline(y_test.mean(), ls='--', c='grey', label='random guess')
ax[0].set_xlabel('Recall'); ax[0].set_ylabel('Precision'); ax[0].set_title('Precision-recall curves (test set)')
ax[0].legend(fontsize=8)

best_name = main.index[0]
pred = (trained[best_name]['pt'] >= trained[best_name]['thr']).astype(int)
ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=['Legit', 'Fraud']).plot(
    ax=ax[1], values_format='d', colorbar=False)
ax[1].set_title(f'{best_name}: confusion matrix at its F1 threshold')
plt.tight_layout(); plt.savefig('figures/pr_curves_and_confusion.png', dpi=150); plt.show()

## 7. Cost-based threshold tuning

A missed fraud costs far more than a false alarm. Set the two costs below (any currency) and the notebook finds the threshold that minimises total cost on the **validation** set, then checks it on the **test** set.

The final model is the tree model with the best validation PR-AUC.

In [ ]:
COST_FN = 10000   # cost of one missed fraud
COST_FP = 100     # cost of one false alarm (analyst review time, customer annoyance)

TREE_CANDIDATES = ['Random Forest', 'XGBoost (weighted)', 'XGBoost + SMOTE', 'XGBoost + undersampling', 'XGBoost (tuned)']
val_pr = {n: average_precision_score(y_val, trained[n]['pv']) for n in TREE_CANDIDATES}
FINAL = max(val_pr, key=val_pr.get)
print('Validation PR-AUC:', {k: round(v, 4) for k, v in val_pr.items()})
print('Final model:', FINAL)

pv, pt = trained[FINAL]['pv'], trained[FINAL]['pt']
yv, yt = y_val.values, y_test.values

def cost_curve(yy, probs):
    fpr, tpr, thr = roc_curve(yy, probs)
    P, N = yy.sum(), (yy == 0).sum()
    return thr, (1 - tpr) * P * COST_FN + fpr * N * COST_FP

thr_grid, cost_grid = cost_curve(yv, pv)
finite = np.isfinite(thr_grid)
THR_COST = float(thr_grid[finite][np.argmin(cost_grid[finite])])
THR_F1 = float(trained[FINAL]['thr'])

def test_cost(thr):
    pred = pt >= thr
    fn = int(((yt == 1) & ~pred).sum()); fp = int(((yt == 0) & pred).sum())
    return fn, fp, fn * COST_FN + fp * COST_FP

summary = []
for label, thr in [('Flag nothing', np.inf), ('Default 0.5', 0.5), ('F1-optimal', THR_F1), ('Cost-optimal', THR_COST)]:
    fn, fp, c = test_cost(thr)
    summary.append({'Strategy': label, 'Threshold': thr, 'Missed frauds': fn, 'False alarms': fp, 'Total cost': c})
cost_table = pd.DataFrame(summary).set_index('Strategy')
display(cost_table)
cost_table.to_csv('cost_table.csv')

## 8. SHAP explanations

SHAP shows how much each feature pushed a prediction toward fraud (positive) or toward legitimate (negative).

In [ ]:
import shap

def tree_of(m):
    return m.steps[-1][1] if hasattr(m, 'steps') else m

final_tree = tree_of(trained[FINAL]['model'])
explainer = shap.TreeExplainer(final_tree)
sample = X_test.sample(min(2000, len(X_test)), random_state=RS)
sv = explainer(sample)
vals = sv.values[:, :, 1] if sv.values.ndim == 3 else sv.values

plt.figure()
shap.summary_plot(vals, sample, show=False)
plt.tight_layout(); plt.savefig('figures/shap_summary.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# Explain one real fraud from the test set (highest predicted probability among true frauds)
fraud_idx = X_test.index[(y_test == 1).values]
top = pd.Series(trained[FINAL]['pt'], index=X_test.index).loc[fraud_idx].idxmax()
one = X_test.loc[[top]]
sv1 = explainer(one)
sv1 = sv1[:, :, 1] if sv1.values.ndim == 3 else sv1
plt.figure()
shap.plots.waterfall(sv1[0], show=False)
plt.tight_layout(); plt.savefig('figures/shap_waterfall_example.png', dpi=150, bbox_inches='tight'); plt.show()
print(one)

## 9. Feature comparison (for the report)

The same XGBoost is trained on three feature sets:

1. **Basic**: amount, hour, transaction type
2. **Pre-transaction**: adds the balances *before* the transaction and ratios or flags derived from them (all known when a transaction is being approved)
3. **Full PaySim columns**: adds the balances *after* the transaction. PaySim cancels fraudulent transactions, so these columns reveal the outcome. Report this row as "all dataset columns, including post-transaction balances".

In [ ]:
def feats(cols):
    return df.loc[idx_train, cols], df.loc[idx_val, cols], df.loc[idx_test, cols]

cmp_runs = [('XGBoost (weighted) - basic features', BASIC_FEATURES, 'basic'),
            ('XGBoost (weighted) - pre-transaction features', PRE_FEATURES, 'pre'),
            ('XGBoost (weighted) - full PaySim columns', FULL_FEATURES, 'full')]
for nm, cols, grp in cmp_runs:
    a_tr, a_va, a_te = feats(cols)
    run_model(nm, make_xgb(scale_pos_weight=spw), a_tr, y_train, a_va, a_te, group=grp)

order = [r[0] for r in cmp_runs] + ['XGBoost (tuned)']
leak = pd.DataFrame(rows).set_index('Model').loc[order, ['Precision', 'Recall', 'F1', 'PR-AUC']].round(4)
leak.index = ['Basic features', 'Pre-transaction features', 'Full PaySim columns (incl. post-transaction balances)',
              f'Tuned XGBoost (FEATURE_MODE = {FEATURE_MODE})']
display(leak)
leak.to_csv('leakage_comparison.csv')

## 10. Save everything the Streamlit app needs

This writes the trained model (`fraud_model.json` for XGBoost, `fraud_model.joblib` otherwise), `model_meta.json` (features, thresholds, versions), `sample_transactions.csv` (a small demo file for the batch tab), the results tables and figures, and zips them all.

In [ ]:
# XGBoost models are saved in portable JSON (works across xgboost versions).
# Other models (Random Forest) fall back to joblib.
if isinstance(final_tree, XGBClassifier):
    final_tree.save_model('fraud_model.json')
    MODEL_FILE, MODEL_FORMAT = 'fraud_model.json', 'xgb_json'
else:
    joblib.dump(final_tree, 'fraud_model.joblib')
    MODEL_FILE, MODEL_FORMAT = 'fraud_model.joblib', 'joblib'

meta = {
    'model_file': MODEL_FILE,
    'model_format': MODEL_FORMAT,
    'model_name': FINAL,
    'features': FEATURES,
    'binary_features': BINARY_FEATURES,
    'feature_mode': FEATURE_MODE,
    'best_params': best_params,
    'risky_types': RISKY,
    'threshold_cost': THR_COST,
    'threshold_f1': THR_F1,
    'cost_fn': COST_FN,
    'cost_fp': COST_FP,
    'sklearn_version': sklearn.__version__,
    'xgboost_version': xgboost.__version__,
}
json.dump(meta, open('model_meta.json', 'w'), indent=2)

# Small demo CSV for the app's batch tab: real test rows, all frauds (up to 40) + random legitimate ones
test_df = df.loc[idx_test, ['type', 'amount', 'step', 'oldbalanceOrg', 'newbalanceOrig',
                            'oldbalanceDest', 'newbalanceDest', 'isFraud']]
fr = test_df[test_df['isFraud'] == 1].head(40)
lg = test_df[test_df['isFraud'] == 0].sample(360, random_state=RS)
pd.concat([fr, lg]).sample(frac=1, random_state=RS).to_csv('sample_transactions.csv', index=False)

for f in ['fraud_artifacts']:
    if os.path.exists(f + '.zip'):
        os.remove(f + '.zip')
os.makedirs('pack', exist_ok=True)
for f in [MODEL_FILE, 'model_meta.json', 'sample_transactions.csv',
          'results_table.csv', 'cost_table.csv', 'leakage_comparison.csv',
          'tuning_results.csv', 'cv_summary.csv']:
    shutil.copy(f, 'pack/' + f)
shutil.copytree('figures', 'pack/figures', dirs_exist_ok=True)
shutil.make_archive('fraud_artifacts', 'zip', 'pack')
print('Saved fraud_artifacts.zip')

try:
    from google.colab import files
    files.download('fraud_artifacts.zip')
except ImportError:
    pass